# Purpose 

This notebook evaluates population level consistency and cross-field relationships in the maternal health dataset. 
The focus is on identifying implausible patterns , distribution anomalies and systematic inconsistencies that are not detectable at individual record level.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("D:/maternal-health-risk-analysis/maternal_health.csv")

In [4]:
type(df) ,df.shape

(pandas.core.frame.DataFrame, (1014, 7))

In [5]:
df["is_error"] = False
df["is_warning"] = False
df["error_reason"] = ""
df["warning_reason"] = ""



In [6]:
df.columns

Index(['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate',
       'RiskLevel', 'is_error', 'is_warning', 'error_reason',
       'warning_reason'],
      dtype='object')

In [7]:
clinical_rules = {
    "age_range": {"min": 10, "max": 55},
    "bp_range": {
        "systolic_min": 70, "systolic_max": 250,
        "diastolic_min": 40, "diastolic_max": 150
    }
}


In [8]:
clinical_rules.keys()

dict_keys(['age_range', 'bp_range'])

In [9]:
age_rule = clinical_rules["age_range"]
mask = (df["Age"] < age_rule["min"]) | (df["Age"] > age_rule["max"])
df.loc[mask, "is_error"] = True
df.loc[mask, "error_reason"] += "Age outside biological range; "


In [10]:
df["is_error"].sum()

np.int64(45)

In [11]:
assert "df" in globals(), "df not loaded – run setup cells"

In [12]:
df["RiskLevel"].value_counts(normalize = True)*100

RiskLevel
low risk     40.039448
mid risk     33.136095
high risk    26.824458
Name: proportion, dtype: float64

In [13]:
df["RiskLevel"].value_counts()

RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64

Risk groups are reasonably distributed, with no single category dominating the dataset. 

In [14]:
df.groupby("RiskLevel")[["SystolicBP", "DiastolicBP", "HeartRate"]].mean()

,SystolicBP,DiastolicBP,HeartRate
RiskLevel,,,
high risk,124.194853,85.073529,76.742647
low risk,105.866995,72.534483,72.770936
mid risk,113.154762,74.232143,74.175595


In [15]:
df.groupby("RiskLevel")["BS"].mean()

RiskLevel
high risk    12.122610
low risk      7.220271
mid risk      7.795744
Name: BS, dtype: float64

Clinical severity increases with risk category

In [16]:
df.groupby("RiskLevel")[["SystolicBP", "DiastolicBP"]].describe()

SystolicBP                                                           \
               count        mean        std   min    25%    50%    75%    max   
RiskLevel                                                                       
high risk      272.0  124.194853  20.227185  83.0  120.0  130.0  140.0  160.0   
low risk       406.0  105.866995  15.894002  70.0   90.0  120.0  120.0  129.0   
mid risk       336.0  113.154762  14.983170  70.0  100.0  120.0  120.0  140.0   

          DiastolicBP                                                        
                count       mean        std   min   25%   50%    75%    max  
RiskLevel                                                                    
high risk       272.0  85.073529  14.112428  60.0  75.0  90.0  100.0  100.0  
low risk        406.0  72.534483  13.054210  49.0  60.0  75.0   80.0  100.0  
mid risk        336.0  74.232143  11.490151  50.0  65.0  75.0   80.0  100.0

While distributions overlap, higher risk groups exhibit consistently higher central tendencies.


In [17]:
df.groupby("RiskLevel")["Age"].describe()
df.groupby("RiskLevel")["BS"].describe()

,count,mean,std,min,25%,50%,75%,max
RiskLevel,,,,,,,,
high risk,272.0,12.122610,4.173525,6.1,7.9,11.0,15.0,19.0
low risk,406.0,7.220271,0.645596,6.0,6.9,7.5,7.5,11.0
mid risk,336.0,7.795744,2.285511,6.0,6.8,7.0,7.8,18.0


In [18]:
df.groupby("RiskLevel")["is_error"].mean() * 100


RiskLevel
high risk    4.044118
low risk     4.926108
mid risk     4.166667
Name: is_error, dtype: float64

In [20]:
df.groupby("RiskLevel")["SystolicBP"].mean()
df[~df["is_error"]].groupby("RiskLevel")["SystolicBP"].mean()
df[~df["is_error"]].groupby("RiskLevel")["DiastolicBP"].mean()

RiskLevel
high risk    85.095785
low risk     72.549223
mid risk     74.136646
Name: DiastolicBP, dtype: float64

In [21]:
df.isnull().mean() * 100


Age               0.0
SystolicBP        0.0
DiastolicBP       0.0
BS                0.0
BodyTemp          0.0
HeartRate         0.0
RiskLevel         0.0
is_error          0.0
is_warning        0.0
error_reason      0.0
warning_reason    0.0
dtype: float64

In [22]:
df.groupby("RiskLevel").apply(lambda x: x.isnull().mean() * 100)


C:\Users\khush\AppData\Local\Temp\ipykernel_15024\911894926.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("RiskLevel").apply(lambda x: x.isnull().mean() * 100)


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel,is_error,is_warning,error_reason,warning_reason
RiskLevel,,,,,,,,,,,
high risk,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
low risk,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mid risk,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
df[
    (df["RiskLevel"] == "low risk") &
    (df["SystolicBP"] > 150)
]


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel,is_error,is_warning,error_reason,warning_reason
